# Week-1 EDA: IEEE-CIS Fraud Detection

**Purpose.** Type-distribution summary, identification of candidate ordinal
features, and a written narrative that orients Module 1 (feature
profiler) and Module 2 (feature segmenter), and that justifies the
contents of [`config/ordinal_features.yaml`](../config/ordinal_features.yaml).

**Data.** Raw CSVs live under `data/raw/` after extracting
`data/ieee-fraud-detection.zip`. The zip and the extracted CSVs are
both gitignored; see the top-level [README.md](../README.md) for the
Kaggle download. Roughly 590,540 transactions, with `train_identity`
joining onto a ~24% subset by `TransactionID`.

**Scope.** This notebook is a one-time orientation pass. It is *not*
the feature profiler (Module 1 builds that as a reproducible script
with cached output to `outputs/feature_profile.json`). Outputs here are
intentionally compact: enough to defend the ordinal annotation, not a
full data dictionary.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

DATA_DIR = Path('..') / 'data' / 'raw'

df_tx = pd.read_csv(DATA_DIR / 'train_transaction.csv')
df_id = pd.read_csv(DATA_DIR / 'train_identity.csv')
df = df_tx.merge(df_id, on='TransactionID', how='left')

print(f'transaction shape: {df_tx.shape}')
print(f'identity shape:    {df_id.shape}')
print(f'merged shape:      {df.shape}')
print(f'isFraud prevalence: {df_tx["isFraud"].mean():.4%}')

## Type distribution

Of 434 columns on the merged frame: ~399 are `float64` (numeric
anonymized features `V1..V339`, time-delta features `D1..D15`, counts
`C1..C14`, plus the address/distance columns), ~31 are string (the
categorical landscape we actually need to encode), and ~4 are `int64`
(`TransactionID`, `isFraud`, `TransactionDT`, `card1`). The encoding
selection problem is essentially the 31 string columns plus a handful
of low-cardinality numerics.

In [ ]:
print(df.dtypes.value_counts())
print()
print(f'overall missingness: {df.isna().mean().mean():.2%}')
print(f'columns with >50% missing: {(df.isna().mean() > 0.5).sum()}')

## Categorical landscape (31 string columns)

Cardinality is bimodal: ~22 columns have ≤ 5 distinct values
(boolean-ish flags like `M1`–`M9`, `id_35`–`id_38`, plus small enums
like `ProductCD`, `card4`, `card6`, `DeviceType`), and a long tail of
high-cardinality identifiers: `id_30` (75 OS strings), `id_31` (130
browser strings), `id_33` (260 resolutions), `DeviceInfo` (1786
device strings), and the email-domain columns (~60 each). High
cardinality + heavy missingness will push Module 3 toward grouped or
frequency encoding for those columns.

In [ ]:
str_cols = df.select_dtypes(include='object').columns.tolist()
rows = []
for c in str_cols:
    rows.append({
        'column': c,
        'nunique': df[c].nunique(dropna=True),
        'missing_rate': df[c].isna().mean(),
    })
cat_summary = pd.DataFrame(rows).sort_values('nunique').reset_index(drop=True)
cat_summary

## Ordinal candidate investigation

Most of the IEEE-CIS categoricals are nominal. Anonymization stripped
semantic ordering from almost everything. The exceptions are columns
whose values *literally encode* an ordering in their string form:

- **`id_34`** — values are `match_status:-1`, `match_status:0`,
  `match_status:1`, `match_status:2`. The integer suffix is the
  ordering.
- **`M4`** — values are `M0`, `M1`, `M2`. The integer suffix again
  suggests a monotone level. The semantic ordering is not documented in
  the IEEE-CIS data dictionary, but the surface form is enough to make
  the ordinal interpretation defensible.

Two close calls we *exclude* (they fall back to the 0.30 ordinal
rubric score downstream — the "uncertain-ordering penalty"):

- **`id_23`** (IP proxy types: HIDDEN / ANONYMOUS / TRANSPARENT). One
  could argue an ordering by transparency, but the direction (more
  concealment ↔ higher fraud risk?) is a modelling judgment, not a
  property of the values. We let the MILP pay the 0.30 penalty rather
  than impose an order we cannot defend.
- **`id_15`** (Found / New / Unknown). A confidence-style sequence is
  plausible, but again the surface form does not commit to one.

Boolean `T`/`F` columns (`M1`–`M3`, `M5`–`M9`, `id_35`–`id_38`) are
*technically* orderable as `{F, T}` but a one-hot and an ordinal
encoding of a 2-value categorical produce the same single column;
annotating them as ordinal would not change Module 3's decision matrix.

In [ ]:
for c in ['id_34', 'M4', 'id_23', 'id_15']:
    print(f'=== {c} ===')
    print(df[c].value_counts(dropna=False).to_string())
    print()

## Conclusions feeding the next deliverables

**Ordinal annotation (`config/ordinal_features.yaml`).** Two entries:

```yaml
id_34:
  - "match_status:-1"
  - "match_status:0"
  - "match_status:1"
  - "match_status:2"
M4:
  - M0
  - M1
  - M2
```

Every other categorical falls back to the 0.30 unlisted score in the
Module 3 rubric. That is the intended behavior — the rubric penalty
encodes our honest uncertainty about ordering.

**Orienting Module 1 (feature profiler).** The profiler emits, per
column: detected type (numeric / categorical), cardinality, a
distribution-shape summary, missingness rate, and mutual information
against `isFraud` via `mutual_info_classif`. Notes from this EDA:

- The merged frame is 590,540 × 434. Running `mutual_info_classif` on
  the full set is expensive. Module 1 should subsample (with seed
  `RANDOM_SEED`); a stratified 10% sample is consistent with the
  Module 3 evaluation policy and produces stable MI rankings.
- The MI value is a *characterization* signal only. LOF (Module 5) and
  the Module 7 MLPs never see `isFraud` as a training target. This is
  the invariant in [CLAUDE.md](../CLAUDE.md) and the module docstring
  must echo it.
- Many columns are >50% missing. Module 1 should report missingness
  rate as a first-class field so Modules 2 and 3 can route those
  columns appropriately (e.g., a missingness indicator + passthrough
  rather than a full encoding sweep).

**Orienting Module 2 (feature segmenter).** The plan pins five
domain-labeled segments: *transaction amount*, *identity/device*,
*behavioral frequency*, *temporal/timing*, *card/account*. The
categorical landscape above maps cleanly: `TransactionAmt` and the
`V*` aggregates → transaction-amount; `DeviceType`, `DeviceInfo`,
`id_30`/`id_31`/`id_33` → identity/device; `C1..C14` (the count
columns) → behavioral-frequency; `TransactionDT` and `D1..D15` →
temporal/timing; `card1..card6`, `addr1`, `addr2` → card/account.
Rule-based assignment will cover most of the frame; k-means on the
feature-profile vector handles the residual.